<a href="https://colab.research.google.com/github/GMISSAGLIA/GM_PyLab/blob/Main/Download_Economic_Data_Part1_BANKIT_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download Economic & Financial Data from Institutional Data Providers
###   **- Part 1 - Banca d'Italia -**

 If you don’t have a paid data provider and you’re looking for core economic statistics, you can rely on institutional sources to build your repository, including:

- [Banca d'Italia - BANKIT](https://a2a.bancaditalia.it/infostat/dataservices/)

- [FRED - Federal Reserve Bank of St. Louis](https://fred.stlouisfed.org/)  ** - ⚠️ You'll need a free FRED API key that you can get here:** [FRED API](https://fred.stlouisfed.org/docs/api/api_key.html)

- [ECB - European Central Bank](https://data.ecb.europa.eu/services/site-directory)

- [EUROSTAT](https://ec.europa.eu/eurostat)

- [BIS](https://stats.bis.org/api-doc/v2/)

In this notebook, I show how to retrieve data from the Bank of Italy Statistical Database (BDS), which provides RESTful endpoints for programmatic (application-to-application, A2A) exports of both data and metadata.
Refer to the official [User Guide](https://www.bancaditalia.it/statistiche/basi-dati/bds/manuale_BDS_en.pdf?language_id=1) for full specifications.

I demonstrate how to use the API and implement a set of utility functions to interact with INFOSTAT, along with several examples.

In [1]:
#install
import subprocess
import sys
# If needed, install dependencies (uncomment to run)
%pip install eurostat pandasdmx ecbdata jsonstat.py
def install_packages():
    packages = [
        'pandasdmx', 'pandasql', 'matplotlib', 'seaborn', 'scikit-learn',
        'statsmodels', 'scipy', 'requests', 'openpyxl','pyarrow', 'yfinance',
        'eurostat', 'ecbdata', 'fredapi', 'jsonstat.py']
    for package in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        except:
            print(f"Package {package} already installed or failed to install")
install_packages()
##############################################################################
#Standard Library
import math
import gzip
import zipfile
import io
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timedelta
import warnings
from functools import reduce
from io import StringIO
import tkinter as tk
from tkinter import Tk, filedialog
from tkinter.filedialog import askopenfilename
from IPython import display
from IPython.display import display # Import the standard display function

# Other Library Imports
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas_datareader.data as web
from pandasql import sqldf
import requests
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.api import VAR, VECM
from statsmodels.tsa.stattools import adfuller, coint
from pathlib import PurePosixPath

from fredapi import Fred
import eurostat

# Define the SQL runner
pysqldf_G = lambda q: sqldf(q, globals())
pysqldf_L = lambda q: sqldf(q, locals())
warnings.filterwarnings("ignore")

##############################################################################
#Constants
EUROSTAT3_URL = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data"
EUROSTAT2_URL = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data"
ECB_SDW = "https://sdw-wsrest.ecb.europa.eu/service/data"
ECB_BULK_DATA ="https://data-api.ecb.europa.eu/service/data"
BANKIT_URL = "https://a2a.bancaditalia.it/infostat/dataservices/export"
DT_START ='2001-12-31'
DT_END = '2024-12-31'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.4/150.4 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of jsonstat-py to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.6/149.6 kB 14.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.6/149.6 kB 14.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 43.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from pathlib import Path

# Base A2A endpoint (can be changed if needed)
BASE = "https://a2a.bancaditalia.it/infostat/dataservices/export"
# Default language/format for new code you add in this notebook
LANG_DEFAULT = "IT"    # or "EN"
FMT_DEFAULT  = "CSV"   # or "XLSX"
# Output folder for any exports
OUTDIR = Path("./output")
OUTDIR.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUTDIR.resolve())

Output folder: /content/output


In [3]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def make_session(retries=3, backoff=0.5, status_forcelist=(429, 500, 502, 503, 504)):
    s = requests.Session()
    retry = Retry(
        total=retries, read=retries, connect=retries,
        backoff_factor=backoff, status_forcelist=status_forcelist,
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://", HTTPAdapter(max_retries=retry))
    return s

SESSION = make_session()
DEFAULT_TIMEOUT = 120
print("Retry-enabled session ready.")

Retry-enabled session ready.



*   **`list_bds_table_codes`**: Retrieves all available BDS table codes by iterating over the publications catalog and extracting each publication’s table identifiers.


In [ ]:
#from pathlib import PurePosixPath
#import re
def list_bds_table_codes(lang:str ="EN", pub_codes:str=None, timeout:int=240, verbose:bool=False,BASE:str = "https://a2a.bancaditalia.it/infostat/dataservices/export" ) -> pd.DataFrame:

    """
    Return all table codes by iterating publications and reading filenames
    from the A2A ZIP (contenttype=DATA).
    """
    # Define a list of publication codes to iterate through
    if pub_codes is None:
        pub_codes = ["STABOL","STAMEN","STACORIS","STAFINRA","STAATER","BAM","MFN","CFI","BOP",  "FPI","FPR","FPE","SDDS","STASDP","SST","IBF","EXSTA" ] # Publication codes from the BdI manual

    rows = []
    #sess = requests.Session()
    sess = make_session()
    # Table code pattern like TRI30529, TDB20207, TFAA0000, ...
    code_re = re.compile(r"^[A-Z]{3,5}\d{3,6}$")

    # Iterate through each publication code
    for pub in pub_codes:
        # Construct the URL for the publication's data ZIP file
        url = f"{BASE}/{lang}/CSV/DATA/PUBLICATION/BANKITALIA/DIFF/{pub}"
        try:
            # Send a GET request to download the ZIP file
            r = sess.get(url, timeout=timeout)
            r.raise_for_status() # Raise an exception for bad status codes

            # Open the downloaded ZIP file
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                names = z.namelist() # Get the list of files in the ZIP
                if verbose:
                    print(pub, "entries:", len(names))
                found = set()
                # Iterate through each file in the ZIP
                for name in names:
                    fname = PurePosixPath(name).name              # keep only filename
                    stem  = fname.split(".")[0].split("_")[0]     # drop ext/suffixes
                    # Check if the filename stem matches the table code pattern
                    if code_re.match(stem):
                        found.add(stem) # Add the found table code to a set

                # Append found table codes with their publication to the rows list
                rows += [{"publication": pub, "table_code": c} for c in sorted(found)]
        except Exception as e:
            # Print a warning if there's an error downloading a publication
            if verbose:
                print(f"[warn] {pub}: {e}")

    # Create a DataFrame from the collected rows, drop duplicates, sort, and reset index
    df = (pd.DataFrame(rows)
            .drop_duplicates()
            .sort_values(["publication", "table_code"])
            .reset_index(drop=True))
    return df

*   **`download_bankit_data`**: Downloads a given BDS table’s DATA, STRUCTURE, DOMAIN, or LEGEND from Banca d’Italia’s INFOSTAT.

In [4]:
import io, zipfile, requests, pandas as pd
from requests.exceptions import HTTPError, Timeout, ConnectionError

def download_bankit_data(
    table_code: str,
    TYPE: str = "DATA",
    BASE: str = "https://a2a.bancaditalia.it/infostat/dataservices/export",
) -> pd.DataFrame:
    """
    Downloads a Banca d'Italia (INFOSTAT) table package via A2A and returns a DataFrame.
    TYPE: 'DATA' | 'STRUCTURE' | 'DOMAIN' | 'LEGEND'
    Note: Some packages contain Excel files or nested ZIPs; this function handles that.
    """
    # ---- validate inputs ----
    if not isinstance(table_code, str) or not table_code.strip():
        raise ValueError("table_code must be a non-empty string.")
    TYPE = (TYPE or "").upper()
    allowed = {"DATA", "STRUCTURE", "DOMAIN", "LEGEND"}
    if TYPE not in allowed:
        raise ValueError(f"TYPE must be one of {sorted(allowed)}")

    # Force CSV package (Italian labels). Change 'IT' to 'EN' if you need English labels.
    url = f"{BASE}/IT/CSV/{TYPE}/CUBE/BANKITALIA/DIFF/{table_code}"
    sess = make_session()

    try:
        r = sess.get(url, timeout=120)
        r.raise_for_status()
    except Timeout as e:
        raise TimeoutError(f"Timeout while downloading {table_code} [{TYPE}] from {url}") from e
    except HTTPError as e:
        status = getattr(e.response, "status_code", None)
        txt = getattr(e.response, "text", "")
        snippet = (txt[:300] + "…") if txt else ""
        raise RuntimeError(f"HTTP {status} for {table_code} [{TYPE}] at {url}. {snippet}") from e
    except ConnectionError as e:
        raise ConnectionError(f"Network error while contacting {url}") from e
    except Exception as e:
        raise RuntimeError(f"Unexpected error requesting {url}: {e}") from e

    # ---- helpers ----
    def _read_bdi_csv(fobj):
        """CSV reader for BDS (tries ';' sep and ',' decimal first)."""
        try:
            return pd.read_csv(fobj, sep=';', decimal=',', dtype=str, low_memory=False)
        except Exception:
            try:
                fobj.seek(0)
            except Exception:
                pass
            return pd.read_csv(fobj, dtype=str, low_memory=False)

    # ---- unzip & parse (CSV first, then Excel fallback, then nested ZIP) ----
    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            csv_names = [n for n in z.namelist() if n.lower().endswith(".csv")]
            if csv_names:
                df = _read_bdi_csv(z.open(csv_names[0]))
            else:
                # Fallback: Excel inside the ZIP
                xlsx_names = [n for n in z.namelist() if n.lower().endswith((".xlsx", ".xls"))]
                if xlsx_names:
                    with z.open(xlsx_names[0]) as f:
                        xls = pd.ExcelFile(io.BytesIO(f.read()))
                    parts = [xls.parse(s, dtype=str).dropna(how="all") for s in xls.sheet_names]
                    df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
                else:
                    # Last chance: some packages embed another ZIP
                    inner_zips = [n for n in z.namelist() if n.lower().endswith(".zip")]
                    if not inner_zips:
                        raise RuntimeError("No CSV/XLSX files found in the returned ZIP.")
                    with zipfile.ZipFile(io.BytesIO(z.read(inner_zips[0]))) as z2:
                        inner_csvs = [n for n in z2.namelist() if n.lower().endswith(".csv")]
                        if not inner_csvs:
                            raise RuntimeError("No CSV files found in nested archive.")
                        df = _read_bdi_csv(z2.open(inner_csvs[0]))
    except zipfile.BadZipFile as e:
        raise zipfile.BadZipFile(f"Invalid ZIP for {table_code} [{TYPE}] from {url}") from e
    except Exception as e:
        raise RuntimeError(f"Failed to read files from package for {table_code} [{TYPE}]: {e}") from e

    # ---- post-processing ----
    if df.empty:
        raise RuntimeError(f"Empty DataFrame for {table_code} [{TYPE}] (no rows).")

    if "OBS_VALUE" in df.columns:
        df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")

    return df



*  **`download_bkt_data`**:Works like **download_bankit_data**, but also supports data_type="ALL", returning a dictionary of DataFrames for DATA, STRUCTURE, DOMAIN, and LEGEND.


In [5]:
import io, re, zipfile, requests, pandas as pd
from requests.exceptions import HTTPError, Timeout, ConnectionError

def download_bkt_data(
    table_code: str,
    TYPE: str = "DATA",
    BASE: str  = "https://a2a.bancaditalia.it/infostat/dataservices/export",
):
    """
    Download a Banca d'Italia (INFOSTAT) table package via A2A.
    Parameters
    ----------
    table_code : str
        e.g. "TRI30529"
    TYPE : str
        One of: 'DATA', 'STRUCTURE', 'DOMAIN', 'LEGEND', 'ALL'
        - 'ALL' may arrive as a ZIP that contains *nested* ZIPs; this function
          will look inside nested archives to find CSV files.
    BASE : str
        A2A base URL.

    Returns
    -------
    pd.DataFrame  (if TYPE != 'ALL')
    dict[str, pd.DataFrame]  (if TYPE == 'ALL'), with keys among:
        {'DATA','STRUCTURE','DOMAIN','LEGEND'}

    Raises
    ------
    ValueError, TimeoutError, ConnectionError, RuntimeError, zipfile.BadZipFile
    """
    # ---- validations ----
    if not isinstance(table_code, str) or not table_code.strip():
        raise ValueError("table_code must be a non-empty string.")
    TYPE = (TYPE or "").upper()
    allowed = {"DATA", "STRUCTURE", "DOMAIN", "LEGEND", "ALL"}
    if TYPE not in allowed:
        raise ValueError(f"TYPE must be one of {sorted(allowed)}")

    # Force CSV format; BDS uses ';' as sep and ',' as decimal in many CSVs
    url = f"{BASE}/IT/CSV/{TYPE}/CUBE/BANKITALIA/DIFF/{table_code}"
    sess = make_session()
    # ---- HTTP download ----
    try:
        r = sess.get(url, timeout=120)
        r.raise_for_status()
    except Timeout as e:
        raise TimeoutError(f"Timeout while downloading {table_code} [{TYPE}] from {url}") from e
    except HTTPError as e:
        code = getattr(e.response, "status_code", None)
        # Short snippet to help debug server-side messages
        text = getattr(e.response, "text", "")
        snippet = (text[:300] + "…") if text else ""
        raise RuntimeError(f"HTTP {code} for {table_code} [{TYPE}] at {url}. {snippet}") from e
    except ConnectionError as e:
        raise ConnectionError(f"Connection error while contacting {url}") from e
    except Exception as e:
        raise RuntimeError(f"Unexpected error requesting {url}: {e}") from e

    # ---- helpers ----
    def _read_bdi_csv(fobj):
        """Robust CSV reader for BDS (tries ';' sep and ',' decimal first)."""
        try:
            return pd.read_csv(fobj, sep=';', decimal=',', dtype=str, low_memory=False)
        except Exception:
            try:
                fobj.seek(0)
            except Exception:
                pass
            return pd.read_csv(fobj, dtype=str, low_memory=False)

    def _iter_csv_entries(zf: zipfile.ZipFile, allow_nested=True):
        """
        Yield (leaf_name, bytes) for every CSV found in the ZIP.
        If allow_nested, also opens any nested .zip entries (one level deep).
        """
        # Top-level CSVs
        for n in zf.namelist():
            if n.lower().endswith(".csv"):
                yield n.rsplit("/", 1)[-1], zf.read(n)

        # Nested ZIPs (common for TYPE='ALL')
        if allow_nested:
            for n in zf.namelist():
                if n.lower().endswith(".zip"):
                    try:
                        with zipfile.ZipFile(io.BytesIO(zf.read(n))) as inner:
                            for leaf, data in _iter_csv_entries(inner, allow_nested=False):
                                yield leaf, data
                    except zipfile.BadZipFile:
                        # Skip entries that aren't valid ZIPs
                        continue

    # ---- unzip & parse ----
    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            entries = list(_iter_csv_entries(z, allow_nested=(TYPE == "ALL")))
    except zipfile.BadZipFile as e:
        raise zipfile.BadZipFile(f"Invalid ZIP for {table_code} [{TYPE}] from {url}") from e

    if not entries:
        raise RuntimeError(
            f"No CSV files found in the package for {table_code} [{TYPE}]. "
            "If you requested TYPE='ALL', the server may have returned nested archives "
            "without CSVs or a different format than expected."
        )

    # ---- return shapes match the original function ----
    if TYPE != "ALL":
        # Use the first CSV (like your original code)
        leaf, data = entries[0]
        df = _read_bdi_csv(io.BytesIO(data))
        if df.empty:
            raise RuntimeError(f"CSV '{leaf}' is empty for {table_code} [{TYPE}].")
        # Cast OBS_VALUE if present
        if "OBS_VALUE" in df.columns:
            df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")
        return df

    # TYPE == 'ALL' → build dict: {'DATA': df, 'STRUCTURE': df, 'DOMAIN': df, 'LEGEND': df}
    # There can be multiple CSVs per category; we’ll concatenate within each bucket.
    buckets = {"DATA": [], "STRUCTURE": [], "DOMAIN": [], "LEGEND": []}
    for leaf, data in entries:
        df = _read_bdi_csv(io.BytesIO(data))
        if df.empty:
            continue
        # Decide which bucket based on filename
        N = leaf.rsplit(".csv", 1)[0].upper()
        if "DOMAIN" in N:
            key = "DOMAIN"
        elif "STRUCTURE" in N:
            key = "STRUCTURE"
        elif "LEGEND" in N:
            key = "LEGEND"
        else:
            key = "DATA"
        buckets[key].append(df)

    # Combine lists to single DataFrames (only keep non-empty)
    out = {}
    for key, frames in buckets.items():
        if not frames:
            continue
        dfk = pd.concat(frames, ignore_index=True)
        if "OBS_VALUE" in dfk.columns:
            dfk["OBS_VALUE"] = pd.to_numeric(dfk["OBS_VALUE"], errors="coerce")
        out[key] = dfk

    if not out:
        raise RuntimeError(
            f"ZIP for {table_code} [ALL] contained CSVs but none could be classified "
            f"as DATA/STRUCTURE/DOMAIN/LEGEND from their filenames."
        )

    return out


In [6]:
#example:
table_code= "TRI30632"
tables = download_bkt_data(table_code, TYPE="ALL")
for df in tables:
  display(tables[df].head())
df = download_bkt_data(table_code, TYPE="DATA")
display(df)

,DATA_OSS,ENTE_SEGN,FENEC,SEDELEG_SOGG,SET_CTP,VALORE,STATUS
0,2025-03-31,3691030,35130163,ITI2,S11,"0,531",NaN
1,2025-03-31,3691030,35130163,ITI2,S13,0,NaN
2,2025-03-31,3691030,35130163,IT,S13,"1,305",NaN
3,2025-03-31,3691030,35130163,ITG1,S12BI7,"0,739",NaN
4,2025-03-31,3691030,351132141,ITI4,S15BI1,1297,NaN


,Cubo,Variabile,Descrizione,Tipologia,Dominio,Valori di dominio
0,TRI30632,DATA_OSS,Data dell'osservazione,VC,TEMPO,Dominio enumerato
1,TRI30632,ENTE_SEGN,Ente segnalante,VC,AZIENDA,3691030
2,TRI30632,FENEC,Fenomeno economico,VC,FENOMECON,Dominio enumerato
3,TRI30632,SEDELEG_SOGG,Sede legale del censito,VC,TERRITORIO,Dominio enumerato
4,TRI30632,SET_CTP,Settore istituzionale della controparte,VC,SETTORIST,Dominio enumerato


,Dominio,Elemento,Descrizione
0,AZIENDA,3691030,Enti segnalanti in Centrale dei rischi
1,FENOMECON,351131433,Flusso trimestrale nuovi prestiti in default r...
2,FENOMECON,351131441,Flusso trimestrale nuovi prestiti in default r...
3,FENOMECON,351132133,Stock prestiti non in default rettificato trim...
4,FENOMECON,351132141,Stock prestiti non in default rettificato trim...


,Tipologia di oggetto,Codice,Descrizione
0,Pubblicazione,STACORIS,Banche e istituzioni finanziarie: condizioni e...
1,Cubo multidimensionale,TRI30632,Flusso trimestrale nuovi prestiti in default r...
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,Definizioni generali,NaN,NaN


,DATA_OSS,ENTE_SEGN,FENEC,SEDELEG_SOGG,SET_CTP,VALORE,STATUS
0,2025-03-31,3691030,35130163,ITI2,S11,"0,531",NaN
1,2025-03-31,3691030,35130163,ITI2,S13,0,NaN
2,2025-03-31,3691030,35130163,IT,S13,"1,305",NaN
3,2025-03-31,3691030,35130163,ITG1,S12BI7,"0,739",NaN
4,2025-03-31,3691030,351132141,ITI4,S15BI1,1297,NaN
...,...,...,...,...,...,...,...
83445,2005-09-30,3691030,35130363,ITI4,S14BI4,"1,439",NaN
83446,2005-09-30,3691030,35130363,ITC4,S15BI1,"0,182",NaN
83447,2005-09-30,3691030,35130363,ITF3,S15BI1,"0,434",NaN
83448,2005-09-30,3691030,35130363,ITG1,S15BI1,"1,162",NaN


**download_bankit_data_all:**
Download all components (DATA, DOMAIN, STRUCTURE, LEGEND) for a table.  Each component is attempted independently. Failures do not stop others.

In [ ]:
def download_bankit_data_all(
    table_code: str,
    BASE: str = "https://a2a.bancaditalia.it/infostat/dataservices/export",
    return_errors: bool = True,
    raise_on_all_fail: bool = False,
) -> dict:
    """
    Download all components (DATA, DOMAIN, STRUCTURE, LEGEND) for a table.
    Each component is attempted independently. Failures do not stop others.
    Returns
    -------
    dict with keys:
      - 'DATA', 'DOMAIN', 'STRUCTURE', 'LEGEND' -> DataFrame or None
      - optional '_errors' -> dict {TYPE: 'error message'} when return_errors=True
    """
    # basic validation
    if not isinstance(table_code, str) or not table_code.strip():
        raise ValueError("table_code must be a non-empty string.")
    types = ("DATA", "DOMAIN", "STRUCTURE", "LEGEND")
    results = {t: None for t in types}
    errors = {}
    for t in types:
        try:
            # forward BASE so caller can override endpoint if needed
            results[t] = download_bankit_data(table_code, TYPE=t, BASE=BASE)
        except Exception as e:
            errors[t] = str(e)
            results[t] = None
    if return_errors and errors:
        results["_errors"] = errors

    if raise_on_all_fail and all(results[t] is None for t in types):
        raise RuntimeError(
            f"All downloads failed for table {table_code}. "
            f"Errors: {errors if errors else 'no details available'}"
        )

    return results
#Example:
table_code= "TRI30632"
tables= download_bankit_data_all(table_code)
for k, v in tables.items():
    print(k)
    display(v.head())

DATA


,DATA_OSS,ENTE_SEGN,FENEC,SEDELEG_SOGG,SET_CTP,VALORE,STATUS
0,2025-03-31,3691030,35130163,ITI2,S11,"0,531",NaN
1,2025-03-31,3691030,35130163,ITI2,S13,0,NaN
2,2025-03-31,3691030,35130163,IT,S13,"1,305",NaN
3,2025-03-31,3691030,35130163,ITG1,S12BI7,"0,739",NaN
4,2025-03-31,3691030,351132141,ITI4,S15BI1,1297,NaN


DOMAIN


,Dominio,Elemento,Descrizione
0,AZIENDA,3691030,Enti segnalanti in Centrale dei rischi
1,FENOMECON,351131433,Flusso trimestrale nuovi prestiti in default r...
2,FENOMECON,351131441,Flusso trimestrale nuovi prestiti in default r...
3,FENOMECON,351132133,Stock prestiti non in default rettificato trim...
4,FENOMECON,351132141,Stock prestiti non in default rettificato trim...


STRUCTURE


,Cubo,Variabile,Descrizione,Tipologia,Dominio,Valori di dominio
0,TRI30632,DATA_OSS,Data dell'osservazione,VC,TEMPO,Dominio enumerato
1,TRI30632,ENTE_SEGN,Ente segnalante,VC,AZIENDA,3691030
2,TRI30632,FENEC,Fenomeno economico,VC,FENOMECON,Dominio enumerato
3,TRI30632,SEDELEG_SOGG,Sede legale del censito,VC,TERRITORIO,Dominio enumerato
4,TRI30632,SET_CTP,Settore istituzionale della controparte,VC,SETTORIST,Dominio enumerato


LEGEND


,Tipologia di oggetto,Codice,Descrizione
0,Pubblicazione,STACORIS,Banche e istituzioni finanziarie: condizioni e...
1,Cubo multidimensionale,TRI30632,Flusso trimestrale nuovi prestiti in default r...
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,Definizioni generali,NaN,NaN


In [ ]:
"""
Seleziono [35130163] Tasso di deterioramento trimestrale dei prestiti - default rettificato: numero affidati
area geografica:italia (SEDELEG_SOGG = "IT")
tipologia di controparte totale controparti al netto delle istituzioni
monetarie e finanziarie (SET_CTP = "SBI42")
"""
#Example:
table_code = "TRI30632"
FENEC="35130163"
SEDELEG_SOGG = "IT"
SET_CTP = "SBI42"


tables= download_bankit_data_all(table_code)
for k, v in tables.items():
  print(k)
  display(v.head())
# Define the SQL query using pandasql syntax
df_data = tables.get("DATA")
query = """
SELECT *
FROM df_data
WHERE FENEC == "35130163"
  AND SEDELEG_SOGG = "IT"
  AND SET_CTP == "SBI42";
"""
# Execute the query using the defined pysqldf_L function
result_df = pysqldf_G(query)
# Display the result
display(result_df.head())

DATA


,DATA_OSS,ENTE_SEGN,FENEC,SEDELEG_SOGG,SET_CTP,VALORE,STATUS
0,2025-03-31,3691030,35130163,ITI2,S11,"0,531",NaN
1,2025-03-31,3691030,35130163,ITI2,S13,0,NaN
2,2025-03-31,3691030,35130163,IT,S13,"1,305",NaN
3,2025-03-31,3691030,35130163,ITG1,S12BI7,"0,739",NaN
4,2025-03-31,3691030,351132141,ITI4,S15BI1,1297,NaN


DOMAIN


,Dominio,Elemento,Descrizione
0,AZIENDA,3691030,Enti segnalanti in Centrale dei rischi
1,FENOMECON,351131433,Flusso trimestrale nuovi prestiti in default r...
2,FENOMECON,351131441,Flusso trimestrale nuovi prestiti in default r...
3,FENOMECON,351132133,Stock prestiti non in default rettificato trim...
4,FENOMECON,351132141,Stock prestiti non in default rettificato trim...


STRUCTURE


,Cubo,Variabile,Descrizione,Tipologia,Dominio,Valori di dominio
0,TRI30632,DATA_OSS,Data dell'osservazione,VC,TEMPO,Dominio enumerato
1,TRI30632,ENTE_SEGN,Ente segnalante,VC,AZIENDA,3691030
2,TRI30632,FENEC,Fenomeno economico,VC,FENOMECON,Dominio enumerato
3,TRI30632,SEDELEG_SOGG,Sede legale del censito,VC,TERRITORIO,Dominio enumerato
4,TRI30632,SET_CTP,Settore istituzionale della controparte,VC,SETTORIST,Dominio enumerato


LEGEND


,Tipologia di oggetto,Codice,Descrizione
0,Pubblicazione,STACORIS,Banche e istituzioni finanziarie: condizioni e...
1,Cubo multidimensionale,TRI30632,Flusso trimestrale nuovi prestiti in default r...
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,Definizioni generali,NaN,NaN


,DATA_OSS,ENTE_SEGN,FENEC,SEDELEG_SOGG,SET_CTP,VALORE,STATUS
0,2025-03-31,3691030,35130163,IT,SBI42,"0,265",None
1,2024-12-31,3691030,35130163,IT,SBI42,"0,289",None
2,2024-09-30,3691030,35130163,IT,SBI42,"0,298",None
3,2024-06-30,3691030,35130163,IT,SBI42,"0,316",None
4,2024-03-31,3691030,35130163,IT,SBI42,"0,303",None


**download_bkt_pub**: downloads a whole publication from Banca d’Italia’s INFOSTAT A2A service (es. STABOL, CFI, BOP)

Support functions:

*  **_read_bdi_csv(fobj)** : Robust CSV reader for BDS
*  **_extract_table_code(filename)** : Extracts table codes (e.g., TRI30529) from file names.
*  **_gather_entries(zf, accept_nested, verbose)**:Iterates through the contents of the ZIP and, optionally, nested ZIPs; returns (leaf_name, bytes) pairs for each CSV/XLSX/XLS found. With verbose=True, it prints a preview of the entries.


In [7]:
#Download a complete publication
# pip install pandas requests openpyxl
import io, re, zipfile, requests, pandas as pd # Added zipfile import
from pathlib import PurePosixPath

BASE = "https://a2a.bancaditalia.it/infostat/dataservices/export"

def _read_bdi_csv(fobj, BASE = "https://a2a.bancaditalia.it/infostat/dataservices/export"):
    """Robust CSV reader for BDS (often ';' sep and ',' decimal)."""
    try:
        return pd.read_csv(fobj, sep=';', decimal=',', dtype=str, low_memory=False)
    except Exception:
        fobj.seek(0)
        return pd.read_csv(fobj, dtype=str, low_memory=False)

def _extract_table_code(filename: str) -> str | None:
    """Guess a table code like TRI30529, TDB20207, TFAA0000 from a filename."""
    m = re.search(r"([A-Z]{3,5}\d{3,6})", filename)
    return m.group(1) if m else None

def _gather_entries(zf: zipfile.ZipFile, accept_nested: bool, verbose: bool):
    """Yield (leaf_name, bytes) for every CSV/XLSX/XLS entry (search nested ZIPs if needed)."""
    names = zf.namelist()
    if verbose:
        print("ZIP entries:", len(names))
        print(*names[:20], sep="\n")
        if len(names) > 20: print("...")

    # First, top-level CSV/XLSX/XLS
    for n in names:
        if n.lower().endswith((".csv", ".xlsx", ".xls")):
            yield PurePosixPath(n).name, zf.read(n)

    # Then, nested ZIPs if asked
    if accept_nested:
        for n in names:
            if n.lower().endswith(".zip"):
                try:
                    with zipfile.ZipFile(io.BytesIO(zf.read(n))) as inner:
                        for leaf, data in _gather_entries(inner, accept_nested=False, verbose=verbose):
                            yield leaf, data
                except zipfile.BadZipFile:
                    continue  # skip odd files

def download_bkt_pub(pub_code: str,
                                TYPE: str = "DATA",     # 'DATA' | 'STRUCTURE' | 'DOMAIN'| 'LEGEND'
                                lang: str = "IT",       # 'IT' | 'EN'
                                fmt: str = "CSV",       # 'CSV' | 'XLSX'
                                as_dict: bool = False,  # return dict {table_code: df}
                                accept_nested: bool = True,
                                timeout: int = 300,
                                verbose: bool = False,
                                BASE = "https://a2a.bancaditalia.it/infostat/dataservices/export"):
    """
    Download an INFOSTAT publication
    Handles nested ZIPs and CSV/XLSX/XLS.

    Returns:
      - dict {table_code: DataFrame} if as_dict=True
      - otherwise, a long DataFrame with ['TableCode','source_file', ...]
    """
    TYPE = TYPE.upper(); lang = lang.upper(); fmt = fmt.upper()
    ALLOWED_TYPE = {"DATA", "STRUCTURE", "DOMAIN", "LEGEND"}
    ALLOWED_FORMAT = {"CSV", "XLSX"}
    if TYPE not in ALLOWED_TYPE:
        raise ValueError("TYPE must be 'DATA', 'STRUCTURE', or 'DOMAIN'")
    if fmt not in ALLOWED_FORMAT:
        raise ValueError("fmt must be 'CSV' or 'XLSX'")
    url = f"{BASE}/{lang}/{fmt}/{TYPE}/PUBLICATION/BANKITALIA/DIFF/{pub_code}"

    sess=make_session()
    r = sess.get(url, timeout=120)
    r.raise_for_status()

    # Open outer ZIP
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        entries = list(_gather_entries(z, accept_nested=accept_nested, verbose=verbose))

    if not entries:
        raise RuntimeError("No CSV/XLSX entries found in publication ZIP (even after checking nested archives).")

    if as_dict:
        out = {}
        for leaf, data in entries:
            tcode = _extract_table_code(leaf) or leaf
            if leaf.lower().endswith(".csv"):
                df = _read_bdi_csv(io.BytesIO(data))
            else:
                xls = pd.ExcelFile(io.BytesIO(data))
                parts = [xls.parse(s, dtype=str).dropna(how="all") for s in xls.sheet_names]
                df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
            out[tcode] = df
        return out

    # Single long DataFrame
    frames = []
    for leaf, data in entries:
        tcode = _extract_table_code(leaf) or leaf
        if leaf.lower().endswith(".csv"):
            df = _read_bdi_csv(io.BytesIO(data))
        else:
            xls = pd.ExcelFile(io.BytesIO(data))
            parts = []
            for s in xls.sheet_names:
                dfi = xls.parse(s, dtype=str).dropna(how="all")
                if not dfi.empty:
                    dfi.insert(0, "_sheet", s)
                    parts.append(dfi)
            df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
        if df.empty:
            continue
        df.insert(0, "source_file", leaf)
        df.insert(0, "TableCode", tcode)
        frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [8]:
#Example:
pub_code = "STACORIS"
df_pub_DATA = download_bkt_pub(pub_code=pub_code, TYPE="DATA", lang="IT", fmt="CSV", as_dict=False, accept_nested=True, verbose=False)
display(df_pub_DATA)

,TableCode,source_file,CLASSE_ACCORD,DATA_OSS,DIVISA1,DURORI,ENTE_SEGN,FENEC,SEDELEG_SOGG,SET_CTP,...,SESSO,CLASSE_IMP_CON,DURORI_STRUMENTO,DESINV,LOC_CTP,CLASSI_PD,TIPO_GARANZIA,SPE_GIU,CLASSE_NUMAFF,TIPTASSO
0,TRI30136,20250627_085651-STACORIS-TRI30136.csv,246,2025-03-31,2,9,3691029,35105539,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TRI30136,20250627_085651-STACORIS-TRI30136.csv,242,2025-03-31,1000,11,1100010,35105539,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,TRI30136,20250627_085651-STACORIS-TRI30136.csv,1011,2025-03-31,2,12,1100010,35105539,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TRI30136,20250627_085651-STACORIS-TRI30136.csv,243,2025-03-31,1000,12,3691029,35105539,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TRI30136,20250627_085651-STACORIS-TRI30136.csv,222,2025-03-31,1000,9,3691006,35105536,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7980083,TRI30268,20250627_085656-STACORIS-TRI30268.csv,NaN,2017-03-31,NaN,NaN,1070001,5836614,NaN,S14BI2,...,NaN,NaN,NaN,NaN,IT,NaN,NaN,NaN,NaN,NaN
7980084,TRI30268,20250627_085656-STACORIS-TRI30268.csv,NaN,2017-03-31,NaN,NaN,1070001,1001302,NaN,S14BI2,...,NaN,NaN,NaN,NaN,IT,NaN,NaN,NaN,NaN,NaN
7980085,TRI30268,20250627_085656-STACORIS-TRI30268.csv,NaN,2017-03-31,NaN,NaN,1070001,5836614,NaN,S11,...,NaN,NaN,NaN,NaN,IT,NaN,NaN,NaN,NaN,NaN
7980086,TRI30268,20250627_085656-STACORIS-TRI30268.csv,NaN,2017-03-31,NaN,NaN,1070001,1001302,NaN,S13,...,NaN,NaN,NaN,NaN,IT,NaN,NaN,NaN,NaN,NaN


**download_bkt_pub_all**: downloads  all components (DATA, DOMAIN, STRUCTURE, LEGEND) of a whole publication from Banca d’Italia’s INFOSTAT A2A service (es. STABOL, CFI, BOP) Each component is attempted independently. Failures do not stop others.

In [ ]:
def download_bkt_pub_all(pub_code: str,
                                TYPE: str = "DATA",     # 'DATA' | 'STRUCTURE' | 'DOMAIN' | 'LEGEND'
                                lang: str = "IT",       # 'IT' | 'EN'
                                fmt: str = "CSV",       # 'CSV' | 'XLSX'
                                as_dict: bool = False,  # return dict {table_code: df}
                                accept_nested: bool = True,
                                timeout: int = 300,
                                verbose: bool = False,
                                BASE = "https://a2a.bancaditalia.it/infostat/dataservices/export"):
   bkt_tables={
       "DATA":download_bkt_pub(pub_code,"DATA", lang, fmt, as_dict,         accept_nested,timeout, verbose, BASE),
       "DOMAIN":download_bkt_pub(pub_code,"DOMAIN", lang, fmt, as_dict,         accept_nested,timeout, verbose, BASE),
       "STRUCTURE":download_bkt_pub(pub_code,"STRUCTURE", lang, fmt, as_dict,   accept_nested,timeout, verbose, BASE),
       "LEGEND":download_bkt_pub(pub_code,"LEGEND", lang, fmt, as_dict,   accept_nested,timeout, verbose, BASE) }
   return bkt_tables


In [ ]:
#Example:
pub_code = "STACORIS"
df_pub_all = download_bkt_pub_all(pub_code=pub_code, TYPE="DATA", lang="IT", fmt="CSV", as_dict=False, accept_nested=True, verbose=False)
display(df_pub_all["DATA"].head())
display(df_pub_all["DOMAIN"].head())
display(df_pub_all["STRUCTURE"].head())
display(df_pub_all["LEGEND"].head())

,TableCode,source_file,CLASSE_ACCORD,DATA_OSS,DIVISA1,DURORI,ENTE_SEGN,FENEC,SEDELEG_SOGG,SET_CTP,...,SESSO,CLASSE_IMP_CON,DURORI_STRUMENTO,DESINV,LOC_CTP,CLASSI_PD,TIPO_GARANZIA,SPE_GIU,CLASSE_NUMAFF,TIPTASSO
0,TRI30136,20250627_085651-STACORIS-TRI30136.csv,246,2025-03-31,2,9,3691029,35105539,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TRI30136,20250627_085651-STACORIS-TRI30136.csv,242,2025-03-31,1000,11,1100010,35105539,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,TRI30136,20250627_085651-STACORIS-TRI30136.csv,1011,2025-03-31,2,12,1100010,35105539,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TRI30136,20250627_085651-STACORIS-TRI30136.csv,243,2025-03-31,1000,12,3691029,35105539,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TRI30136,20250627_085651-STACORIS-TRI30136.csv,222,2025-03-31,1000,9,3691006,35105536,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,TableCode,source_file,Dominio,Elemento,Descrizione
0,20250630_080147-DOMAIN-STACORIS-MULTICUBE.csv,20250630_080147-DOMAIN-STACORIS-MULTICUBE.csv,ATECO,000000,Informazione non prevista o non applicabile
1,20250630_080147-DOMAIN-STACORIS-MULTICUBE.csv,20250630_080147-DOMAIN-STACORIS-MULTICUBE.csv,ATECO,1000055,Prodotti chimici e farmaceutici
2,20250630_080147-DOMAIN-STACORIS-MULTICUBE.csv,20250630_080147-DOMAIN-STACORIS-MULTICUBE.csv,ATECO,1000060,Fabbricazione di autoveicoli e altri mezzi di ...
3,20250630_080147-DOMAIN-STACORIS-MULTICUBE.csv,20250630_080147-DOMAIN-STACORIS-MULTICUBE.csv,ATECO,1000061,"Industrie alimentari, delle bevande e del tabacco"
4,20250630_080147-DOMAIN-STACORIS-MULTICUBE.csv,20250630_080147-DOMAIN-STACORIS-MULTICUBE.csv,ATECO,1000062,"Industrie tessili, abbigliamento e articoli i..."


,TableCode,source_file,Cubo,Variabile,Descrizione,Tipologia,Dominio,Valori di dominio
0,20250630_080150-STRUCTURE-STACORIS-MULTICUBE.csv,20250630_080150-STRUCTURE-STACORIS-MULTICUBE.csv,TRI30021_35105532,ATECO_CTP,Attività economica della controparte (ateco 2007),VC,ATECO,Dominio enumerato
1,20250630_080150-STRUCTURE-STACORIS-MULTICUBE.csv,20250630_080150-STRUCTURE-STACORIS-MULTICUBE.csv,TRI30021_35105532,DATA_OSS,Data dell'osservazione,VC,TEMPO,Dominio enumerato
2,20250630_080150-STRUCTURE-STACORIS-MULTICUBE.csv,20250630_080150-STRUCTURE-STACORIS-MULTICUBE.csv,TRI30021_35105532,DIVISA1,Divisa,VC,VALUTISO,Dominio enumerato
3,20250630_080150-STRUCTURE-STACORIS-MULTICUBE.csv,20250630_080150-STRUCTURE-STACORIS-MULTICUBE.csv,TRI30021_35105532,DURORI,Durata originaria dell'operazione,VC,DURATA,Dominio enumerato
4,20250630_080150-STRUCTURE-STACORIS-MULTICUBE.csv,20250630_080150-STRUCTURE-STACORIS-MULTICUBE.csv,TRI30021_35105532,ENTE_SEGN,Ente segnalante,VC,AZIENDA,Dominio enumerato


,TableCode,source_file,Tipologia di oggetto,Codice,Descrizione
0,20250630_080151-LEGEND-STACORIS-MULTICUBE.csv,20250630_080151-LEGEND-STACORIS-MULTICUBE.csv,Pubblicazione,STACORIS,Banche e istituzioni finanziarie: condizioni e...
1,20250630_080151-LEGEND-STACORIS-MULTICUBE.csv,20250630_080151-LEGEND-STACORIS-MULTICUBE.csv,Cubo multidimensionale,TRI30021_35105532,Prestiti (escluse sofferenze): accordato opera...
2,20250630_080151-LEGEND-STACORIS-MULTICUBE.csv,20250630_080151-LEGEND-STACORIS-MULTICUBE.csv,Cubo multidimensionale,TRI30021_35105533,Prestiti (escluse sofferenze): utilizzato
3,20250630_080151-LEGEND-STACORIS-MULTICUBE.csv,20250630_080151-LEGEND-STACORIS-MULTICUBE.csv,Cubo multidimensionale,TRI30021_35105536,Prestiti (escluse sofferenze): importo garantito
4,20250630_080151-LEGEND-STACORIS-MULTICUBE.csv,20250630_080151-LEGEND-STACORIS-MULTICUBE.csv,Cubo multidimensionale,TRI30021_35105539,Prestiti (escluse sofferenze): sconfinamento


In [ ]:
"""
Seleziono [35130163] Tasso di deterioramento trimestrale dei prestiti - default rettificato: numero affidati
area geografica:italia (SEDELEG_SOGG = "IT")
tipologia di controparte totale controparti al netto delle istituzioni
monetarie e finanziarie (SET_CTP = "SBI42")"""
table_code = "TRI30632"
FENEC="35130163"
SEDELEG_SOGG = "IT"
SET_CTP = "SBI42"
df = df_pub_all['DATA']
display(df[(df['TableCode']==table_code) & (df["FENEC"]==FENEC) & (df["SEDELEG_SOGG"]==SEDELEG_SOGG) & (df["SET_CTP"]==SET_CTP)])

,TableCode,source_file,CLASSE_ACCORD,DATA_OSS,DIVISA1,DURORI,ENTE_SEGN,FENEC,SEDELEG_SOGG,SET_CTP,...,SESSO,CLASSE_IMP_CON,DURORI_STRUMENTO,DESINV,LOC_CTP,CLASSI_PD,TIPO_GARANZIA,SPE_GIU,CLASSE_NUMAFF,TIPTASSO
7691222,TRI30632,20250627_085651-STACORIS-TRI30632.csv,NaN,2025-03-31,NaN,NaN,3691030,35130163,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7691426,TRI30632,20250627_085651-STACORIS-TRI30632.csv,NaN,2024-12-31,NaN,NaN,3691030,35130163,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7692507,TRI30632,20250627_085651-STACORIS-TRI30632.csv,NaN,2024-09-30,NaN,NaN,3691030,35130163,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7694352,TRI30632,20250627_085651-STACORIS-TRI30632.csv,NaN,2024-06-30,NaN,NaN,3691030,35130163,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7695374,TRI30632,20250627_085651-STACORIS-TRI30632.csv,NaN,2024-03-31,NaN,NaN,3691030,35130163,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7769289,TRI30632,20250627_085651-STACORIS-TRI30632.csv,NaN,2006-09-30,NaN,NaN,3691030,35130163,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7769836,TRI30632,20250627_085651-STACORIS-TRI30632.csv,NaN,2006-06-30,NaN,NaN,3691030,35130163,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7770987,TRI30632,20250627_085651-STACORIS-TRI30632.csv,NaN,2006-03-31,NaN,NaN,3691030,35130163,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7772077,TRI30632,20250627_085651-STACORIS-TRI30632.csv,NaN,2005-12-31,NaN,NaN,3691030,35130163,IT,SBI42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Below, we retrieve the list of all available INFOSTAT (BdI) table codes:**

In [ ]:
#We get all the codes
df_codes = list_bds_table_codes()
display(df_codes)
#then we find a specific table code
df_codes[df_codes['table_code']=='TRI30529']

,publication,table_code
0,BAM,AGGM0100
1,BAM,AGGM0200
2,BAM,AGGM0300
3,BAM,AGGM0400
4,BAM,AGGM0500
...,...,...
923,STASDP,TSP30000
924,STASDP,TSP30100
925,STASDP,TSP60000
926,STASDP,TSP60100


,publication,table_code
795,STACORIS,TRI30529


In [ ]:
table_code="TRI30529"
try:
    x = download_bankit_data_all(table_code)
    # If the download is successful, you can optionally display or process 'x'
    display(x['DOMAIN'])
    display(x['DATA'])
    display(x['STRUCTURE'])
    display(x['LEGEND'])
except Exception as e:
    print(f"❌ Error downloading data for table code {table_code}: {e}")


,Dominio,Elemento,Descrizione
0,ATECO,1000055,Prodotti chimici e farmaceutici
1,ATECO,1000060,Fabbricazione di autoveicoli e altri mezzi di ...
2,ATECO,1000061,"Industrie alimentari, delle bevande e del tabacco"
3,ATECO,1000062,"Industrie tessili, abbigliamento e articoli i..."
4,ATECO,1000063,"Carta, articoli di carta e prodotti della stampa"
...,...,...,...
66,TIPODATO,902,Rapporto
67,TIPODATO,953,Valore elementare
68,UNMIS,EUR,Euro
69,UNMIS,NP,Numero puro


,ATECO_CTP,CLASSE_UTILIZZ,DATA_OSS,ENTE_SEGN,FENEC,SEDELEG_SOGG,SET_CTP,VALORE,STATUS
0,26,1005,2025-03-31,3691030,35120363,ITI,SBI25,0,NaN
1,26,1004,2025-03-31,3691030,35120363,ITH,SBI25,"0,539",NaN
2,25,1006,2025-03-31,3691030,35120363,ITG,SBI25,0,NaN
3,C,1005,2025-03-31,3691030,35120363,ITH,SBI25,"0,38",NaN
4,25,9904,2025-03-31,3691030,351121433,ITF,SBI25,8883707,NaN
...,...,...,...,...,...,...,...,...,...
500807,1000061,1006,1996-03-31,3691030,35120163,IT,SBI25,"0,581",NaN
500808,1000061,1006,1996-03-31,3691030,35120163,ITC,SBI25,"0,497",NaN
500809,1000061,1006,1996-03-31,3691030,35120363,ITC,SBI25,"0,325",NaN
500810,1000061,1006,1996-03-31,3691030,35120363,ITF,SBI25,"1,767",NaN


,Cubo,Variabile,Descrizione,Tipologia,Dominio,Valori di dominio
0,TRI30529,ATECO_CTP,Attività economica della controparte (ateco 2007),VC,ATECO,Dominio enumerato
1,TRI30529,CLASSE_UTILIZZ,Classe di grandezza del fido globale utilizzato,VC,CLAGRAND,Dominio enumerato
2,TRI30529,DATA_OSS,Data dell'osservazione,VC,TEMPO,Dominio enumerato
3,TRI30529,ENTE_SEGN,Ente segnalante,VC,AZIENDA,3691030
4,TRI30529,FENEC,Fenomeno economico,VC,FENOMECON,Dominio enumerato
5,TRI30529,SEDELEG_SOGG,Sede legale del censito,VC,TERRITORIO,Dominio enumerato
6,TRI30529,SET_CTP,Settore istituzionale della controparte,VC,SETTORIST,SBI25
7,TRI30529,VALORE,Valore,MS,NUMBER,Numero
8,TRI30529,ANNOBASE,Anno base,AT,AUTODESCRIPTIVE,0
9,TRI30529,FONTE,Fonte dei dati,AT,FONTE,BICR


,Tipologia di oggetto,Codice,Descrizione
0,Pubblicazione,STACORIS,Banche e istituzioni finanziarie: condizioni e...
1,Cubo multidimensionale,TRI30529,Flusso trimestrale nuove sofferenze rettificat...
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,Definizioni generali,NaN,NaN
5,Pubblicazione,Insieme di tavole dati pubblicate contestualmente,NaN
6,Tavola statistica,Insieme di cubi multidimensionali o di serie s...,NaN
7,Famiglia,Insieme di dati con la medesima struttura,NaN
8,Cubo multidimensionale,Insieme di dati con la medesima struttura,NaN
9,Serie storica,Sequenza di dati misurati ad intervalli di tem...,NaN


In [ ]:
#Create a dictionary to store all data (DATA, DOMAIN, STRUCTURE, LEGEND) for each table code
dict_bankit_data = {}
# Iterate through the table codes and download all associated data using download_bankit_data_all
for table_code in df_codes['table_code']:
    try:
        # Use the download_bankit_data_all function to get all data types for the table code
        all_data_for_table = download_bankit_data_all(table_code)
        # Store the dictionary of dataframes for this table code
        dict_bankit_data[table_code] = all_data_for_table
        #print(f"✅ Downloaded all data for table code: {table_code}")
    except Exception as e:
        print(f"❌ Error downloading all data for table code {table_code}: {e}")


In [ ]:
# You can now access the data for a specific table code and type like this:
data_for_TRI30529 = dict_bankit_data.get('TRI30529', {})
if data_for_TRI30529 is not None:
  display(data_for_TRI30529.get('DATA').head())
  display(data_for_TRI30529.get('DOMAIN').head())
  display(data_for_TRI30529.get('STRUCTURE').head())
  display(data_for_TRI30529.get('LEGEND').head())
else:
  print("Data for TRI30529 not found.")

,ATECO_CTP,CLASSE_UTILIZZ,DATA_OSS,ENTE_SEGN,FENEC,SEDELEG_SOGG,SET_CTP,VALORE,STATUS
0,26,1005,2025-03-31,3691030,35120363,ITI,SBI25,0,NaN
1,26,1004,2025-03-31,3691030,35120363,ITH,SBI25,"0,539",NaN
2,25,1006,2025-03-31,3691030,35120363,ITG,SBI25,0,NaN
3,C,1005,2025-03-31,3691030,35120363,ITH,SBI25,"0,38",NaN
4,25,9904,2025-03-31,3691030,351121433,ITF,SBI25,8883707,NaN


,Dominio,Elemento,Descrizione
0,ATECO,1000055,Prodotti chimici e farmaceutici
1,ATECO,1000060,Fabbricazione di autoveicoli e altri mezzi di ...
2,ATECO,1000061,"Industrie alimentari, delle bevande e del tabacco"
3,ATECO,1000062,"Industrie tessili, abbigliamento e articoli i..."
4,ATECO,1000063,"Carta, articoli di carta e prodotti della stampa"


,Cubo,Variabile,Descrizione,Tipologia,Dominio,Valori di dominio
0,TRI30529,ATECO_CTP,Attività economica della controparte (ateco 2007),VC,ATECO,Dominio enumerato
1,TRI30529,CLASSE_UTILIZZ,Classe di grandezza del fido globale utilizzato,VC,CLAGRAND,Dominio enumerato
2,TRI30529,DATA_OSS,Data dell'osservazione,VC,TEMPO,Dominio enumerato
3,TRI30529,ENTE_SEGN,Ente segnalante,VC,AZIENDA,3691030
4,TRI30529,FENEC,Fenomeno economico,VC,FENOMECON,Dominio enumerato


,Tipologia di oggetto,Codice,Descrizione
0,Pubblicazione,STACORIS,Banche e istituzioni finanziarie: condizioni e...
1,Cubo multidimensionale,TRI30529,Flusso trimestrale nuove sofferenze rettificat...
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,Definizioni generali,NaN,NaN
